In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [2]:
import os
import pickle
import pandas as pd
import numpy as np
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import random
from itertools import product
from typing import List, Tuple, Optional
from IPython.display import clear_output
%matplotlib inline

os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

# desktop version
data_folder = "G:\Meu Drive\mestrado_final_files\data\\"
models_folder = "G:\Meu Drive\mestrado_final_files\models\\SpentTimeModel_ExtraInfo_State\\"

# google colab version
# data_folder = "drive/MyDrive/mestrado_final_files/data/"
# models_folder = "drive/MyDrive/mestrado_final_files/models/SpentTimeModel_ExtraInfo_State/"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
device

device(type='cpu')

In [4]:
pd.set_option("display.max_columns", None)
pd.set_option("display.expand_frame_repr", False)

In [5]:
BATCH_SIZE = 1024
EPOCHS = 50
VOCAB_SIZE = 601
TIME_SIZE = 301
STATE_EMBEDDING_DIM = 16
STATE_SIZE = 225

run_models = 1

In [6]:
%run 01_functions_training.ipynb
#%run drive/MyDrive/mestrado_final_files/notebooks/01_functions_training.ipynb

In [7]:
with open(data_folder + 'data_time_features.pkl', 'rb') as f:
    df_time_features = pickle.load(f)

In [8]:
with open(data_folder + 'dict_set_covariables.pkl', 'rb') as f:
    dict_set_covariables = pickle.load(f)

In [9]:
list_seasons = ['2018-19', '2019-20', '2020-21', '2021-22', '2022-23']
list_context = [5]
list_embedding_layer_dim = [32, 64]
list_num_neurons = [50]
list_num_layers = [3, 5]
list_dropout = [0, 0.3]
list_layernorm = [False, True]
list_set_covariables = [1, 2]

In [10]:
all_combinations = list(product(list_seasons, list_context, list_embedding_layer_dim, list_num_neurons, list_num_layers,
                                list_dropout, list_layernorm, list_set_covariables))

In [11]:
model_mask_name = "{}_seasondata{}_context{}_embedding{}_numlayers{}_numneurons{}_{}_layernorm{}_setcovariables{}_parameters.pth"

In [12]:
i = 0
for comb in all_combinations:
    i += 1
    print(i)
    season, CONTEXT_SIZE, EMBEDDING_LAYER_SIZE, NUM_HIDDEN_NEURONS, NUM_LAYERS, dropout_rate, layer_norm, key_covariables = comb

    print(comb)
    index_covariables = dict_set_covariables["index"][key_covariables]
    covariables = dict_set_covariables["covariables"][key_covariables]

    # ================== DADOS TREINO/DEV - MODELO TOTAL ==================
    df_ = df_time_features.query("season == @season & season_split == 'train'")

    trn_tkn = torch.tensor(np.asarray(df_["token_features"].to_list()), dtype = torch.int64, device = device)
    trn_num = torch.tensor(np.asarray(df_["numerical_features"].to_list()), dtype = torch.float32, device = device)
    trn_stt = torch.tensor(np.asarray(df_["token_state"].to_list()), dtype = torch.int64, device = device)
    trn_y   = torch.tensor(np.asarray(df_["target_time"].to_list()), dtype = torch.int64, device = device)
    trn_mask = torch.tensor(build_logit_mask_spent_time_models(df_), dtype = torch.bool, device = device)

    df_ = df_time_features.query("season == @season & season_split == 'dev'")

    dev_tkn = torch.tensor(np.asarray(df_["token_features"].to_list()), dtype = torch.int64, device = device)
    dev_num = torch.tensor(np.asarray(df_["numerical_features"].to_list()), dtype = torch.float32, device = device)
    dev_stt = torch.tensor(np.asarray(df_["token_state"].to_list()), dtype = torch.int64, device = device)
    dev_y   = torch.tensor(np.asarray(df_["target_time"].to_list()), dtype = torch.int64, device = device)
    dev_mask = torch.tensor(build_logit_mask_spent_time_models(df_), dtype = torch.bool, device = device)

    del df_
    # =======================================================================

    tp_dropout = "nodropout" if dropout_rate == 0 else "dropout" + str(dropout_rate).replace(".", "")

    time_model = SpentTimeModel_ExtraInfo_State(
        context_size = CONTEXT_SIZE,
        vocab_size = VOCAB_SIZE,
        time_size = TIME_SIZE,
        embedding_dim = EMBEDDING_LAYER_SIZE,
        numerical_input_dim = len(index_covariables),
        state_vocab_size = STATE_SIZE,
        state_embedding_dim = STATE_EMBEDDING_DIM,
        num_hidden_neurons = NUM_HIDDEN_NEURONS,
        num_layers = NUM_LAYERS,
        activation_function = nn.GELU,
        dropout_rate = dropout_rate,
        use_layer_norm = layer_norm)

    str_season = season.replace("-", "")

    model_filename = model_mask_name\
    .format(time_model.__class__.__name__, str_season, CONTEXT_SIZE, EMBEDDING_LAYER_SIZE, NUM_LAYERS, NUM_HIDDEN_NEURONS, tp_dropout, str(layer_norm), key_covariables)

    results_filename = "results_"  + model_filename.replace("_parameters.pth", ".pkl")

    if results_filename in os.listdir(models_folder):
        print("Already fitted.")
        continue

    time_model, trnloss, devloss = train_spent_time_model_extra_info_state(
        model = time_model,
        train_token_data = trn_tkn[:, -CONTEXT_SIZE:],
        train_numerical_data = trn_num[:, index_covariables],
        train_state_data = trn_stt,
        train_targets = trn_y,
        train_mask_data = trn_mask,
        batch_size = BATCH_SIZE,
        n_epochs = EPOCHS,
        lr = 0.002,
        val_token_data = dev_tkn[:, -CONTEXT_SIZE:],
        val_numerical_data = dev_num[:, index_covariables],
        val_state_data = dev_stt,
        val_targets = dev_y,
        val_mask_data = dev_mask,
        params_optimizer = {'factor': 0.5, 'threshold': 0, 'patience': 1},
        device = device)

    df_importance = permutation_importance_ngram(
        model = time_model,
        target = dev_y,
        tokens = dev_tkn[:, -CONTEXT_SIZE:],
        state_tokens = dev_stt,
        numerical_features = dev_num[:, index_covariables],
        names_numerical_features = covariables,
        num_repeats = 10,
        mask_data = dev_mask)

    print(df_importance)

    output = time_model(trn_tkn[:, -CONTEXT_SIZE:], trn_num[:, index_covariables], trn_stt)
    output = output.masked_fill(trn_mask, -1e9)
    baseline_loss = F.cross_entropy(output, trn_y).item()

    print(f"train data loss: {baseline_loss}")

    torch.save(time_model.state_dict(), models_folder + model_filename)

    dict_results = {
        "class_model": time_model.__class__.__name__,
        "model_filename": model_filename,
        "token_context_size": CONTEXT_SIZE,
        "token_embedding_dim": EMBEDDING_LAYER_SIZE,
        "token_state_embedding_dim": STATE_EMBEDDING_DIM,
        "state_embedding_vocab_size": STATE_SIZE,
        "training_epochs": EPOCHS,
        "numerical_covariables": covariables,
        "training_batch_loss": np.asarray(trnloss),
        "train_loss": baseline_loss,
        "dev_loss": np.asarray(devloss),
        "num_parameters": get_num_parameters(time_model),
        "variable_importance": df_importance}

    with open(models_folder + results_filename, 'wb') as f:
        pickle.dump(dict_results, f)

    clear_output(wait = True)


61
('2019-20', 5, 64, 50, 5, 0.3, False, 1)


KeyboardInterrupt: 